# Anomaly Detection in Exoplanet Light Curves

This notebook demonstrates how AstroAgent can detect transit signals and anomalies in stellar brightness data. We'll use the agent to fetch real light curves and search for planetary transit signatures.

**Techniques covered:**
- Sigma-clipping for statistical outlier detection
- Box-fitting transit search (simplified BLS)
- Phase-folded transit visualization

In [ ]:
from astroagent import AstroAgent

agent = AstroAgent(verbose=True)

## Step 1: Fetch a light curve

In [ ]:
result = agent.ask(
    "Fetch the Kepler light curve for TRAPPIST-1 and analyze it for "
    "any anomalies or transit signals. Use both sigma-clipping and "
    "transit search methods."
)

## Step 2: Work with the raw data directly

You can also use the tools directly without the agent for more control.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astroagent.tools.anomaly_detector import DetectAnomalies

# Generate synthetic data with a transit
np.random.seed(42)
n_points = 1000
time = np.linspace(0, 20, n_points)
flux = 1.0 + np.random.normal(0, 0.001, n_points)

# Inject periodic transits
period = 6.1  # days (similar to TRAPPIST-1 e)
transit_depth = 0.004
transit_duration = 0.3

for t_center in np.arange(period/2, 20, period):
    mask = np.abs(time - t_center) < transit_duration / 2
    flux[mask] -= transit_depth

# Run anomaly detection
detector = DetectAnomalies()
results = detector.execute(
    flux_values=flux.tolist(),
    time_values=time.tolist(),
    method="both",
    sigma_threshold=3.0,
)

print(results["summary"])
print(f"\nTransit candidates found: {len(results.get('transit_candidates', []))}")
for tc in results.get("transit_candidates", [])[:3]:
    print(f"  - Time {tc['start_time']:.2f}-{tc['end_time']:.2f}: depth={tc['depth_percent']:.3f}%, significance={tc['significance_sigma']:.1f}σ")

In [ ]:
# Plot the light curve with detected anomalies
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), height_ratios=[3, 1])

ax1.scatter(time, flux, s=1, alpha=0.4, color='#378ADD', label='Flux')
ax1.set_ylabel('Relative Flux')
ax1.set_title('Synthetic Light Curve with Injected Transits')
ax1.legend()

# Mark outliers
if results.get('sigma_clip', {}).get('outliers'):
    outlier_times = [o['time'] for o in results['sigma_clip']['outliers']]
    outlier_flux = [o['flux'] for o in results['sigma_clip']['outliers']]
    ax1.scatter(outlier_times, outlier_flux, s=20, color='#E24B4A', zorder=5, label='Outliers')
    ax1.legend()

# Mark transit candidates
for tc in results.get('transit_candidates', [])[:5]:
    ax2.axvspan(tc['start_time'], tc['end_time'], alpha=0.3, color='#22c55e')

ax2.scatter(time, flux, s=1, alpha=0.4, color='#378ADD')
ax2.set_xlabel('Time (BJD)')
ax2.set_ylabel('Flux')
ax2.set_title('Transit Candidates (green regions)')

plt.tight_layout()
plt.show()

## Step 3: Ask the agent to interpret

Let the agent reason about what these detections mean scientifically.

In [ ]:
result = agent.ask(
    "I found transit signals with a depth of about 0.4% and a period of ~6.1 days "
    "in stellar brightness data. What kind of planet could cause this? "
    "How does this compare to known exoplanets?"
)
print(result.text)